In [ ]:
!pip install datasets scikit-learn regex

1. Setup

In [ ]:
import pandas as pd
import numpy as np
import json
import re
import regex
import unicodedata
import os

from sklearn.model_selection import train_test_split

print("Setup completed successfully!")

Setup completed successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving mypos-ver.3.0.txt to mypos-ver.3.0.txt


# 2. Load Dataset

In [ ]:
with open(filename, "r", encoding="utf-8") as f:
    for i in range(10):
        line = f.readline()
        print(f"Line {i+1}: {repr(line)}")

Line 1: 'ဒီ/adj ဆေး/n က/ppm ၁၀၀/num ရာခိုင်နှုန်း/n ဆေးဘက်ဝင်/adj အပင်/n များ/part မှ/ppm ဖော်စပ်/v ထား/part တာ/part ဖြစ်/v တယ်/ppm ။/punc\n'
Line 2: 'အသစ်/n ဝယ်/v ထား/part တဲ့/part ဆွယ်တာ/n က/ppm အသီး/n ထ/v နေ/part ပါ/part ပေါ့/part ။/punc\n'
Line 3: 'မ/part ကျန်းမာ/v လျှင်/conj နတ်/n|ဆရာ/n ထံ/ppm မေးမြန်း/v ၍/conj သက်ဆိုင်ရာ/n နတ်/n တို့/part အား/ppm ပူဇော်ပသ/v ရ/part သည်/ppm ။/punc\n'
Line 4: 'ပေဟိုင်/n|ဥယျာဉ်/n ။/punc\n'
Line 5: 'နဝမ/adj အိပ်မက်/n ကောသလ/n|မင်း/n|အိပ်မက်/n ၉/num နက်ရှိုင်း/adj ကျယ်ဝန်း/adj သော/part ရေကန်/n ကြီး/adj တစ်/tn ခု/part တွင်/ppm သတ္တဝါ/n တို့/part ဆင်း/v ၍/conj ရေသောက်/v ကြ/part ၏/ppm ။/punc\n'
Line 6: 'အပြင်ပန်း/n ကြည့်/v ရင်/conj ခက်/adj သလို/part ထင်/v ရ/part ပေမယ့်/conj တကယ့်/adj လက်တွေ့/n အခြေအနေ/n က/ppm တော့/part အဲဒီ/pron လို/ppm မ/part ဟုတ်/v ပါ/part ဘူး/part ။/punc\n'
Line 7: '8/fw bit/fw ပုံရိပ်/n တစ်/tn ခု/part သည်/ppm 256/fw color/fw သို့မဟုတ်/conj gray/fw scale/fw များ/part ကို/ppm အထောက်အကူ/n ပြု/v သည်/ppm ။/punc\n'
Line 8: 'ကိုရီးယား/n ဝတ်စု

3. Inspect Dataset

In [ ]:
print("Number of sentences:", len(lines))

print("\nFirst sentence:")
print(lines[0])

print("\nLast sentence:")
print(lines[-1])

Number of sentences: 43196

First sentence:
ဒီ/adj ဆေး/n က/ppm ၁၀၀/num ရာခိုင်နှုန်း/n ဆေးဘက်ဝင်/adj အပင်/n များ/part မှ/ppm ဖော်စပ်/v ထား/part တာ/part ဖြစ်/v တယ်/ppm ။/punc


Last sentence:
New/fw York/fw Times/fw သတင်းစာ/n က/ppm ဝေဖန်ရေးသမား/n တွေ/part က/ppm တော့/part ပေါ့ပေါ့ပါးပါး/n အရောင်အသွေး/n စုံစုံလင်လင်/n နဲ့/conj လူထု/n ကို/ppm ဧည့်ခံ/v မယ့်/ppm ဇာတ်ကား/n လို့/part ပြော/v တာ/part ပဲ/part ။/punc



4. Parse and Clean Dataset

In [ ]:
import json

print(
    json.dumps(
        records[0],
        ensure_ascii=False,
        indent=2
    )
)

{
  "id": "sent_00001",
  "source_text": "ဒီ/adj ဆေး/n က/ppm ၁၀၀/num ရာခိုင်နှုန်း/n ဆေးဘက်ဝင်/adj အပင်/n များ/part မှ/ppm ဖော်စပ်/v ထား/part တာ/part ဖြစ်/v တယ်/ppm ။/punc",
  "tokens": [
    "ဒီ",
    "ဆေး",
    "က",
    "၁၀၀",
    "ရာခိုင်နှုန်း",
    "ဆေးဘက်ဝင်",
    "အပင်",
    "များ",
    "မှ",
    "ဖော်စပ်",
    "ထား",
    "တာ",
    "ဖြစ်",
    "တယ်",
    "။"
  ],
  "pos_tags": [
    "adj",
    "n",
    "ppm",
    "num",
    "n",
    "adj",
    "n",
    "part",
    "ppm",
    "v",
    "part",
    "part",
    "v",
    "ppm",
    "punc"
  ],
  "compound_flags": [
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false,
    false
  ],
  "text": "ဒီဆေးက၁၀၀ရာခိုင်နှုန်းဆေးဘက်ဝင်အပင်များမှဖော်စပ်ထားတာဖြစ်တယ်။"
}


In [ ]:
print(records[0]["text"])
print(records[0]["tokens"])
print(records[0]["pos_tags"])

ဒီဆေးက၁၀၀ရာခိုင်နှုန်းဆေးဘက်ဝင်အပင်များမှဖော်စပ်ထားတာဖြစ်တယ်။
['ဒီ', 'ဆေး', 'က', '၁၀၀', 'ရာခိုင်နှုန်း', 'ဆေးဘက်ဝင်', 'အပင်', 'များ', 'မှ', 'ဖော်စပ်', 'ထား', 'တာ', 'ဖြစ်', 'တယ်', '။']
['adj', 'n', 'ppm', 'num', 'n', 'adj', 'n', 'part', 'ppm', 'v', 'part', 'part', 'v', 'ppm', 'punc']


In [ ]:
for record in records:
    record["text"] = "".join(record["tokens"])

In [ ]:
mismatch_count = 0

for record in records:
    if len(record["tokens"]) != len(record["pos_tags"]):
        mismatch_count += 1

print("Token/POS mismatches:", mismatch_count)

Token/POS mismatches: 0


In [ ]:
for item in invalid_lines[:20]:
    print(
        item["line_number"],
        repr(item["content"])
    )

In [ ]:
print("Original sentences:", len(lines))
print("Successfully parsed:", len(records))
print("Invalid sentences:", len(invalid_lines))

Original sentences: 43196
Successfully parsed: 43196
Invalid sentences: 0


In [ ]:
records = []
invalid_lines = []

for i, line in enumerate(lines):

    parsed = parse_mypos_line(line)

    if parsed is None:
        invalid_lines.append({
            "line_number": i + 1,
            "content": line
        })
        continue

    record = {
        "id": f"sent_{i+1:05d}",
        "source_text": line.strip(),
        "tokens": parsed["tokens"],
        "pos_tags": parsed["pos_tags"],
        "compound_flags": parsed["compound_flags"]
    }

    records.append(record)

In [ ]:
parsed = parse_mypos_line(lines[0])

print("Tokens:", len(parsed["tokens"]))
print("POS tags:", len(parsed["pos_tags"]))

print(
    "Valid:",
    len(parsed["tokens"]) == len(parsed["pos_tags"])
)

Tokens: 15
POS tags: 15
Valid: True


In [ ]:
{
    "tokens": [
        "မ",
        "ကျန်းမာ",
        "လျှင်",
        "နတ်",
        "ဆရာ",
        ...
    ],

    "pos_tags": [
        "part",
        "v",
        "conj",
        "n",
        "n",
        ...
    ]
}

{'tokens': ['မ', 'ကျန်းမာ', 'လျှင်', 'နတ်', 'ဆရာ', Ellipsis],
 'pos_tags': ['part', 'v', 'conj', 'n', 'n', Ellipsis]}

In [ ]:
sample = lines[2]

print("Original:")
print(sample)

parsed = parse_mypos_line(sample)

print("\nParsed:")
print(parsed)

Original:
မ/part ကျန်းမာ/v လျှင်/conj နတ်/n|ဆရာ/n ထံ/ppm မေးမြန်း/v ၍/conj သက်ဆိုင်ရာ/n နတ်/n တို့/part အား/ppm ပူဇော်ပသ/v ရ/part သည်/ppm ။/punc


Parsed:
{'tokens': ['မ', 'ကျန်းမာ', 'လျှင်', 'နတ်', 'ဆရာ', 'ထံ', 'မေးမြန်း', '၍', 'သက်ဆိုင်ရာ', 'နတ်', 'တို့', 'အား', 'ပူဇော်ပသ', 'ရ', 'သည်', '။'], 'pos_tags': ['part', 'v', 'conj', 'n', 'n', 'ppm', 'v', 'conj', 'n', 'n', 'part', 'ppm', 'v', 'part', 'ppm', 'punc'], 'compound_flags': [False, False, False, True, True, False, False, False, False, False, False, False, False, False, False, False]}


In [ ]:
def parse_mypos_line(line):
    line = line.strip()

    if not line:
        return None

    tokens = []
    pos_tags = []
    compound_flags = []

    chunks = line.split()

    for chunk in chunks:

        # myPOS can contain compounds such as:
        # နတ်/n|ဆရာ/n
        parts = chunk.split("|")

        is_compound = len(parts) > 1

        for part in parts:

            if "/" not in part:
                return None

            word, tag = part.rsplit("/", 1)

            word = word.strip()
            tag = tag.strip()

            if not word or not tag:
                return None

            tokens.append(word)
            pos_tags.append(tag)
            compound_flags.append(is_compound)

    return {
        "tokens": tokens,
        "pos_tags": pos_tags,
        "compound_flags": compound_flags
    }

In [ ]:
tokens = ["ဒီ", "ဆေး", "က", "၁၀၀"]
pos_tags = ["adj", "n", "ppm", "num"]

In [ ]:
print("First raw line:")
print(repr(lines[0]))

First raw line:
'ဒီ/adj ဆေး/n က/ppm ၁၀၀/num ရာခိုင်နှုန်း/n ဆေးဘက်ဝင်/adj အပင်/n များ/part မှ/ppm ဖော်စပ်/v ထား/part တာ/part ဖြစ်/v တယ်/ppm ။/punc\n'


In [ ]:
for i, line in enumerate(lines[:20]):
    print(f"{i+1}: {line}")

1: ဒီ/adj ဆေး/n က/ppm ၁၀၀/num ရာခိုင်နှုန်း/n ဆေးဘက်ဝင်/adj အပင်/n များ/part မှ/ppm ဖော်စပ်/v ထား/part တာ/part ဖြစ်/v တယ်/ppm ။/punc

2: အသစ်/n ဝယ်/v ထား/part တဲ့/part ဆွယ်တာ/n က/ppm အသီး/n ထ/v နေ/part ပါ/part ပေါ့/part ။/punc

3: မ/part ကျန်းမာ/v လျှင်/conj နတ်/n|ဆရာ/n ထံ/ppm မေးမြန်း/v ၍/conj သက်ဆိုင်ရာ/n နတ်/n တို့/part အား/ppm ပူဇော်ပသ/v ရ/part သည်/ppm ။/punc

4: ပေဟိုင်/n|ဥယျာဉ်/n ။/punc

5: နဝမ/adj အိပ်မက်/n ကောသလ/n|မင်း/n|အိပ်မက်/n ၉/num နက်ရှိုင်း/adj ကျယ်ဝန်း/adj သော/part ရေကန်/n ကြီး/adj တစ်/tn ခု/part တွင်/ppm သတ္တဝါ/n တို့/part ဆင်း/v ၍/conj ရေသောက်/v ကြ/part ၏/ppm ။/punc

6: အပြင်ပန်း/n ကြည့်/v ရင်/conj ခက်/adj သလို/part ထင်/v ရ/part ပေမယ့်/conj တကယ့်/adj လက်တွေ့/n အခြေအနေ/n က/ppm တော့/part အဲဒီ/pron လို/ppm မ/part ဟုတ်/v ပါ/part ဘူး/part ။/punc

7: 8/fw bit/fw ပုံရိပ်/n တစ်/tn ခု/part သည်/ppm 256/fw color/fw သို့မဟုတ်/conj gray/fw scale/fw များ/part ကို/ppm အထောက်အကူ/n ပြု/v သည်/ppm ။/punc

8: ကိုရီးယား/n ဝတ်စုံ/n မှာ/ppm ပန်း/n ဒီဇိုင်း/n နဲ့/conj အဝါရောင်/n က/ppm လိုက်ဖ

In [ ]:
with open(filename, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total lines:", len(lines))

Total lines: 43196


In [ ]:
filename = list(uploaded.keys())[0]

print("Dataset file:", filename)

Dataset file: mypos-ver.3.0.txt


5. Normalize Burmese Unicode

In [ ]:
normalization_changes = 0

for record in records:
    for token in record["tokens"]:

        normalized = unicodedata.normalize(
            "NFC",
            token
        )

        if token != normalized:
            normalization_changes += 1

print(
    "Tokens changed by NFC normalization:",
    normalization_changes
)

Tokens changed by NFC normalization: 0


In [ ]:
for record in records:

    record["text"] = normalize_unicode(
        record["text"]
    )

    record["tokens"] = [
        normalize_unicode(token)
        for token in record["tokens"]
    ]

In [ ]:
import unicodedata

def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

6. Inspect and Validate POS Tags

In [ ]:
print(sorted(pos_counter.keys()))

['abb', 'adj', 'adv', 'conj', 'fw', 'int', 'n', 'num', 'part', 'ppm', 'pron', 'punc', 'sb', 'tn', 'v']


In [ ]:
rare_tags = {
    tag: count
    for tag, count in pos_counter.items()
    if count < 50
}

print("Rare POS tags:")

for tag, count in rare_tags.items():
    print(tag, count)

Rare POS tags:


In [ ]:
from collections import Counter

pos_counter = Counter()

for record in records:
    pos_counter.update(record["pos_tags"])

print("Unique POS tags:", len(pos_counter))

print("\nPOS tags:")
for tag, count in pos_counter.most_common():
    print(tag, count)

Unique POS tags: 15

POS tags:
part 135267
n 122892
ppm 86490
v 84080
punc 54108
pron 20413
conj 17808
adj 16430
adv 10711
num 5942
tn 5844
fw 3228
int 672
abb 360
sb 272


7. Standardize POS Tags

In [ ]:
print(records[0]["pos_tags"])
print(records[0]["pos_ids"])

['adj', 'n', 'ppm', 'num', 'n', 'adj', 'n', 'part', 'ppm', 'v', 'part', 'part', 'v', 'ppm', 'punc']
[1, 6, 9, 7, 6, 1, 6, 8, 9, 14, 8, 8, 14, 9, 11]


In [ ]:
for record in records:
    record["pos_ids"] = [
        pos2id[tag]
        for tag in record["pos_tags"]
    ]

In [ ]:
pos2id = {
    tag: idx
    for idx, tag in enumerate(POS_TAGS)
}

id2pos = {
    idx: tag
    for tag, idx in pos2id.items()
}

print("POS to ID:")
print(pos2id)

POS to ID:
{'abb': 0, 'adj': 1, 'adv': 2, 'conj': 3, 'fw': 4, 'int': 5, 'n': 6, 'num': 7, 'part': 8, 'ppm': 9, 'pron': 10, 'punc': 11, 'sb': 12, 'tn': 13, 'v': 14}


In [ ]:
POS_TAGS = sorted(pos_counter.keys())

print("Standard POS tag set:")
print(POS_TAGS)
print("Number of tags:", len(POS_TAGS))

Standard POS tag set:
['abb', 'adj', 'adv', 'conj', 'fw', 'int', 'n', 'num', 'part', 'ppm', 'pron', 'punc', 'sb', 'tn', 'v']
Number of tags: 15


8. Create Segmentation Labels

In [ ]:
reconstruction_errors = 0

for record in records:
    reconstructed = "".join(record["seg_units"])

    if reconstructed != record["text"]:
        reconstruction_errors += 1

print(
    "Segmentation reconstruction errors:",
    reconstruction_errors
)

Segmentation reconstruction errors: 0


In [ ]:
seg_mismatch_count = 0

for record in records:
    if len(record["seg_units"]) != len(record["seg_labels"]):
        seg_mismatch_count += 1

print(
    "Segmentation unit/label mismatches:",
    seg_mismatch_count
)

Segmentation unit/label mismatches: 0


In [ ]:
print("Text:")
print(records[0]["text"])

print("\nTokens:")
print(records[0]["tokens"])

print("\nSegmentation units:")
print(records[0]["seg_units"])

print("\nSegmentation labels:")
print(records[0]["seg_labels"])

Text:
ဒီဆေးက၁၀၀ရာခိုင်နှုန်းဆေးဘက်ဝင်အပင်များမှဖော်စပ်ထားတာဖြစ်တယ်။

Tokens:
['ဒီ', 'ဆေး', 'က', '၁၀၀', 'ရာခိုင်နှုန်း', 'ဆေးဘက်ဝင်', 'အပင်', 'များ', 'မှ', 'ဖော်စပ်', 'ထား', 'တာ', 'ဖြစ်', 'တယ်', '။']

Segmentation units:
['ဒီ', 'ဆေ', 'း', 'က', '၁', '၀', '၀', 'ရ', 'ာ', 'ခို', 'င်', 'နှု', 'န်', 'း', 'ဆေ', 'း', 'ဘ', 'က်', 'ဝ', 'င်', 'အ', 'ပ', 'င်', 'မျ', 'ာ', 'း', 'မှ', 'ဖေ', 'ာ်', 'စ', 'ပ်', 'ထ', 'ာ', 'း', 'တ', 'ာ', 'ဖြ', 'စ်', 'တ', 'ယ်', '။']

Segmentation labels:
['B', 'B', 'I', 'B', 'B', 'I', 'I', 'B', 'I', 'I', 'I', 'I', 'I', 'I', 'B', 'I', 'I', 'I', 'I', 'I', 'B', 'I', 'I', 'B', 'I', 'I', 'B', 'B', 'I', 'I', 'I', 'B', 'I', 'I', 'B', 'I', 'B', 'I', 'B', 'I', 'B']


In [ ]:
for record in records:
    units, labels = create_segmentation_labels(
        record["tokens"]
    )

    record["seg_units"] = units
    record["seg_labels"] = labels

In [ ]:
def create_segmentation_labels(tokens):
    units = []
    labels = []

    for token in tokens:
        token_units = graphemes(token)

        for i, unit in enumerate(token_units):
            units.append(unit)

            if i == 0:
                labels.append("B")
            else:
                labels.append("I")

    return units, labels

In [ ]:
sample_word = records[0]["tokens"][0]

print("Word:", sample_word)
print("Units:", graphemes(sample_word))

Word: ဒီ
Units: ['ဒီ']


In [ ]:
def graphemes(text):
    return regex.findall(r"\X", text)

In [ ]:
import regex

9. Check Dataset Statistics

In [ ]:
total_sentences = len(records)

total_tokens = sum(
    len(record["tokens"])
    for record in records
)

total_seg_units = sum(
    len(record["seg_units"])
    for record in records
)

avg_tokens = total_tokens / total_sentences

print("Total sentences:", total_sentences)
print("Total tokens:", total_tokens)
print("Total segmentation units:", total_seg_units)
print("Average tokens per sentence:", round(avg_tokens, 2))

Total sentences: 43196
Total tokens: 564517
Total segmentation units: 1401537
Average tokens per sentence: 13.07


In [ ]:
token_lengths = [
    len(record["tokens"])
    for record in records
]

print(
    "Shortest sentence:",
    min(token_lengths),
    "tokens"
)

print(
    "Longest sentence:",
    max(token_lengths),
    "tokens"
)

Shortest sentence: 1 tokens
Longest sentence: 423 tokens


In [ ]:
print(
    "Average sentence length:",
    round(np.mean(token_lengths), 2)
)

NameError: name 'np' is not defined

In [ ]:
import numpy as np

In [ ]:
print(
    "Average sentence length:",
    round(np.mean(token_lengths), 2)
)

Average sentence length: 13.07


In [ ]:
import numpy as np

token_lengths = [
    len(record["tokens"])
    for record in records
]

print("Shortest sentence:", min(token_lengths), "tokens")
print("Longest sentence:", max(token_lengths), "tokens")
print("Average sentence length:", round(np.mean(token_lengths), 2))

Shortest sentence: 1 tokens
Longest sentence: 423 tokens
Average sentence length: 13.07


In [ ]:
print(
    "Average sentence length:",
    round(np.mean(token_lengths), 2)
)

Average sentence length: 13.07


In [ ]:
pos_distribution = pd.DataFrame(
    pos_counter.most_common(),
    columns=["pos_tag", "count"]
)

pos_distribution

NameError: name 'pd' is not defined

In [ ]:
import pandas as pd

In [ ]:
pos_distribution = pd.DataFrame(
    pos_counter.most_common(),
    columns=["pos_tag", "count"]
)

pos_distribution

,pos_tag,count
0,part,135267
1,n,122892
2,ppm,86490
3,v,84080
4,punc,54108
5,pron,20413
6,conj,17808
7,adj,16430
8,adv,10711
9,num,5942


In [ ]:
text_counter = Counter(
    record["text"]
    for record in records
)

duplicate_texts = {
    text: count
    for text, count in text_counter.items()
    if count > 1
}

print(
    "Number of duplicated sentence texts:",
    len(duplicate_texts)
)

Number of duplicated sentence texts: 743


In [ ]:
duplicate_instances = sum(
    count - 1
    for count in duplicate_texts.values()
)

print(
    "Extra duplicate records:",
    duplicate_instances
)

Extra duplicate records: 1274


In [ ]:
for text, count in list(duplicate_texts.items())[:10]:
    print(count, text)

2 ခင်ဗျားအသက်ဘယ်လောက်ရှိပြီလဲ။
2 နားလည်ပါသလား။
10 ဒီနေ့ဘာနေ့လဲ။
4 ဒီနေ့ဘယ်နှရက်နေ့လဲ။
2 ကံကောင်းပါစေ။
2 လေဆိပ်ကိုဘယ်အချိန်သွားသင့်သလဲ။
2 မရှိဘူး။
12 ဘာဖြစ်လို့လဲ။
2 ဘာစာအုပ်လဲ။
2 တခြားအားကစားမကြိုက်ဘူးလား။


In [ ]:
unique_records = []
seen = set()

for record in records:
    key = (
        record["text"],
        tuple(record["tokens"]),
        tuple(record["pos_tags"])
    )

    if key not in seen:
        seen.add(key)
        unique_records.append(record)

print("Before:", len(records))
print("After:", len(unique_records))

Before: 43196
After: 42052


In [ ]:
records = unique_records

10. Split Train / Validation / Test

In [ ]:
train_data, temp_data = train_test_split(
    records,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

NameError: name 'train_test_split' is not defined

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train_data, temp_data = train_test_split(
    records,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

In [ ]:
print("Total:", len(records))
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Total: 42052
Train: 33641
Validation: 4205
Test: 4206


In [ ]:
total = len(records)

print(
    "Train %:",
    round(len(train_data) / total * 100, 2)
)

print(
    "Validation %:",
    round(len(validation_data) / total * 100, 2)
)

print(
    "Test %:",
    round(len(test_data) / total * 100, 2)
)

Train %: 80.0
Validation %: 10.0
Test %: 10.0


In [ ]:
train_texts = {
    record["text"]
    for record in train_data
}

val_texts = {
    record["text"]
    for record in validation_data
}

test_texts = {
    record["text"]
    for record in test_data
}

print(
    "Train ↔ Validation overlap:",
    len(train_texts & val_texts)
)

print(
    "Train ↔ Test overlap:",
    len(train_texts & test_texts)
)

print(
    "Validation ↔ Test overlap:",
    len(val_texts & test_texts)
)

Train ↔ Validation overlap: 17
Train ↔ Test overlap: 20
Validation ↔ Test overlap: 2


In [ ]:
from collections import Counter

text_counter = Counter(
    record["text"]
    for record in records
)

duplicate_texts = {
    text: count
    for text, count in text_counter.items()
    if count > 1
}

print("Duplicated sentence texts:", len(duplicate_texts))

extra_duplicates = sum(
    count - 1
    for count in duplicate_texts.values()
)

print("Extra duplicate records:", extra_duplicates)

Duplicated sentence texts: 129
Extra duplicate records: 130


In [ ]:
for text, count in list(duplicate_texts.items())[:20]:
    print("Count:", count)
    print(text)
    print("-" * 50)

Count: 2
ဒီနေ့ဘာနေ့လဲ။
--------------------------------------------------
Count: 2
ဒီနေ့ဘယ်နှရက်နေ့လဲ။
--------------------------------------------------
Count: 2
ကံကောင်းပါစေ။
--------------------------------------------------
Count: 2
နေသိပ်မကောင်းဘူး။
--------------------------------------------------
Count: 2
အခန်းသန့်ရှင်းရေးလုပ်ပေးပါ။
--------------------------------------------------
Count: 2
ဟုတ်ကဲ့။ဒီဘက်ကိုကြွပါ။
--------------------------------------------------
Count: 2
ဟုတ်ကဲ့၊ကောင်းပါတယ်။ကျေးဇူးတင်ပါတယ်။
--------------------------------------------------
Count: 2
အေးလား။
--------------------------------------------------
Count: 2
ဟုတ်ကဲ့ပါဆရာ။
--------------------------------------------------
Count: 2
အများကြီးကျေးဇူးတင်ပါတယ်။
--------------------------------------------------
Count: 2
ကျေးဇူးတင်ပါတယ်။
--------------------------------------------------
Count: 2
ထိုဗိုလ်မှူးကြီးသည်နောင်အခါ၌‘မြန်မာပြည်လွတ်လပ်ရေးကိုကူညီရန်နှင့်တရုတ်မြန်မာလမ်းမကြီးကိုပိတ်ဆို့ရန်’ရည်ရွယ်ချက်မျ

In [ ]:
records = unique_records

In [ ]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(
    records,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

In [ ]:
train_texts = {record["text"] for record in train_data}
val_texts = {record["text"] for record in validation_data}
test_texts = {record["text"] for record in test_data}

print("Train ↔ Validation overlap:",
      len(train_texts & val_texts))

print("Train ↔ Test overlap:",
      len(train_texts & test_texts))

print("Validation ↔ Test overlap:",
      len(val_texts & test_texts))

Train ↔ Validation overlap: 17
Train ↔ Test overlap: 20
Validation ↔ Test overlap: 2


In [ ]:
unique_by_text = []
seen_texts = set()

for record in records:
    if record["text"] not in seen_texts:
        seen_texts.add(record["text"])
        unique_by_text.append(record)

print("Before text deduplication:", len(records))
print("After text deduplication:", len(unique_by_text))

records = unique_by_text

Before text deduplication: 42052
After text deduplication: 41922


In [ ]:
from sklearn.model_selection import train_test_split

train_data, temp_data = train_test_split(
    records,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Total records:", len(records))
print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Total records: 41922
Train: 33537
Validation: 4192
Test: 4193


In [ ]:
train_texts = {
    record["text"]
    for record in train_data
}

val_texts = {
    record["text"]
    for record in validation_data
}

test_texts = {
    record["text"]
    for record in test_data
}

print(
    "Train ↔ Validation overlap:",
    len(train_texts & val_texts)
)

print(
    "Train ↔ Test overlap:",
    len(train_texts & test_texts)
)

print(
    "Validation ↔ Test overlap:",
    len(val_texts & test_texts)
)

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0


In [ ]:
total = len(records)

print(
    "Train %:",
    round(len(train_data) / total * 100, 2)
)

print(
    "Validation %:",
    round(len(validation_data) / total * 100, 2)
)

print(
    "Test %:",
    round(len(test_data) / total * 100, 2)
)

Train %: 80.0
Validation %: 10.0
Test %: 10.0


11. Save Processed Files

In [ ]:
import os
import json

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/samples", exist_ok=True)
os.makedirs("reports", exist_ok=True)

In [ ]:
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in data:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

In [ ]:
save_jsonl(
    train_data,
    "data/processed/train.jsonl"
)

save_jsonl(
    validation_data,
    "data/processed/validation.jsonl"
)

save_jsonl(
    test_data,
    "data/processed/test.jsonl"
)

In [1]:
with open(
    "data/processed/pos2id.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        pos2id,
        f,
        ensure_ascii=False,
        indent=2
    )

with open(
    "data/processed/id2pos.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        id2pos,
        f,
        ensure_ascii=False,
        indent=2
    )

print("POS mappings saved!")

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/pos2id.json'

In [2]:
import os
import json
import pandas as pd
import numpy as np

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/samples", exist_ok=True)
os.makedirs("reports", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [3]:
import os
import json
import pandas as pd
import numpy as np

os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/samples", exist_ok=True)
os.makedirs("reports", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [4]:
print(os.listdir("data"))
print(os.listdir("data/processed"))
print(os.listdir("data/samples"))
print(os.listdir("reports"))

['samples', 'processed']
[]
[]
[]


In [6]:
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in data:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

print("save_jsonl function ready!")

save_jsonl function ready!


In [9]:
save_jsonl(
    train_data,
    "data/processed/train.jsonl"
)

save_jsonl(
    validation_data,
    "data/processed/validation.jsonl"
)

save_jsonl(
    test_data,
    "data/processed/test.jsonl"
)

print("Train, validation, and test datasets saved!")

NameError: name 'train_data' is not defined

In [8]:
import pandas as pd
import numpy as np
import json
import re
import regex
import unicodedata
import os

from collections import Counter
from sklearn.model_selection import train_test_split

print("Setup completed successfully!")

Setup completed successfully!


In [10]:
from google.colab import files

uploaded = files.upload()

Saving mypos-ver.3.0.txt to mypos-ver.3.0.txt


In [12]:
filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total lines:", len(lines))

Total lines: 43196


In [13]:
def parse_mypos_line(line):
    line = line.strip()

    if not line:
        return None

    tokens = []
    pos_tags = []
    compound_flags = []

    chunks = line.split()

    for chunk in chunks:
        parts = chunk.split("|")
        is_compound = len(parts) > 1

        for part in parts:
            if "/" not in part:
                return None

            word, tag = part.rsplit("/", 1)

            word = word.strip()
            tag = tag.strip()

            if not word or not tag:
                return None

            tokens.append(word)
            pos_tags.append(tag)
            compound_flags.append(is_compound)

    return {
        "tokens": tokens,
        "pos_tags": pos_tags,
        "compound_flags": compound_flags
    }

In [14]:
records = []
invalid_lines = []

for i, line in enumerate(lines):
    parsed = parse_mypos_line(line)

    if parsed is None:
        invalid_lines.append({
            "line_number": i + 1,
            "content": line
        })
        continue

    record = {
        "id": f"sent_{i+1:05d}",
        "source_text": line.strip(),
        "tokens": parsed["tokens"],
        "pos_tags": parsed["pos_tags"],
        "compound_flags": parsed["compound_flags"]
    }

    record["text"] = "".join(record["tokens"])

    records.append(record)

print("Records:", len(records))
print("Invalid:", len(invalid_lines))

Records: 43196
Invalid: 0


In [16]:
def normalize_unicode(text):
    return unicodedata.normalize("NFC", text)

for record in records:
    record["text"] = normalize_unicode(record["text"])
    record["tokens"] = [
        normalize_unicode(token)
        for token in record["tokens"]
    ]

In [17]:
pos_counter = Counter()

for record in records:
    pos_counter.update(record["pos_tags"])

POS_TAGS = sorted(pos_counter.keys())

pos2id = {
    tag: idx
    for idx, tag in enumerate(POS_TAGS)
}

id2pos = {
    idx: tag
    for tag, idx in pos2id.items()
}

for record in records:
    record["pos_ids"] = [
        pos2id[tag]
        for tag in record["pos_tags"]
    ]

In [18]:
def graphemes(text):
    return regex.findall(r"\X", text)

def create_segmentation_labels(tokens):
    units = []
    labels = []

    for token in tokens:
        token_units = graphemes(token)

        for i, unit in enumerate(token_units):
            units.append(unit)

            if i == 0:
                labels.append("B")
            else:
                labels.append("I")

    return units, labels

In [19]:
for record in records:
    units, labels = create_segmentation_labels(
        record["tokens"]
    )

    record["seg_units"] = units
    record["seg_labels"] = labels

In [20]:
unique_records = []
seen = set()

for record in records:
    key = (
        record["text"],
        tuple(record["tokens"]),
        tuple(record["pos_tags"])
    )

    if key not in seen:
        seen.add(key)
        unique_records.append(record)

records = unique_records

In [21]:
unique_by_text = []
seen_texts = set()

for record in records:
    if record["text"] not in seen_texts:
        seen_texts.add(record["text"])
        unique_by_text.append(record)

records = unique_by_text

print("Final deduplicated records:", len(records))

Final deduplicated records: 41922


In [22]:
train_data, temp_data = train_test_split(
    records,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

validation_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

Train: 33537
Validation: 4192
Test: 4193


In [23]:
train_texts = {record["text"] for record in train_data}
validation_texts = {record["text"] for record in validation_data}
test_texts = {record["text"] for record in test_data}

print(
    "Train ↔ Validation:",
    len(train_texts & validation_texts)
)

print(
    "Train ↔ Test:",
    len(train_texts & test_texts)
)

print(
    "Validation ↔ Test:",
    len(validation_texts & test_texts)
)

Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0


In [24]:
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/samples", exist_ok=True)
os.makedirs("reports", exist_ok=True)

print("Folders created successfully!")

Folders created successfully!


In [25]:
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for record in data:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False
                ) + "\n"
            )

In [26]:
save_jsonl(
    train_data,
    "data/processed/train.jsonl"
)

save_jsonl(
    validation_data,
    "data/processed/validation.jsonl"
)

save_jsonl(
    test_data,
    "data/processed/test.jsonl"
)

print("Train, validation, and test datasets saved!")

Train, validation, and test datasets saved!


In [27]:
print(
    os.path.exists(
        "data/processed/train.jsonl"
    )
)

print(
    os.path.exists(
        "data/processed/validation.jsonl"
    )
)

print(
    os.path.exists(
        "data/processed/test.jsonl"
    )
)

True
True
True


In [28]:
save_jsonl(
    train_data,
    "data/processed/train.jsonl"
)

save_jsonl(
    validation_data,
    "data/processed/validation.jsonl"
)

save_jsonl(
    test_data,
    "data/processed/test.jsonl"
)

print("Train, validation, and test datasets saved!")

Train, validation, and test datasets saved!


In [29]:
print("Train exists:",
      os.path.exists("data/processed/train.jsonl"))

print("Validation exists:",
      os.path.exists("data/processed/validation.jsonl"))

print("Test exists:",
      os.path.exists("data/processed/test.jsonl"))

Train exists: True
Validation exists: True
Test exists: True


In [30]:
with open(
    "data/processed/pos2id.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        pos2id,
        f,
        ensure_ascii=False,
        indent=2
    )

with open(
    "data/processed/id2pos.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        id2pos,
        f,
        ensure_ascii=False,
        indent=2
    )

print("POS mappings saved!")

POS mappings saved!


In [31]:
print(
    os.path.exists("data/processed/pos2id.json")
)

print(
    os.path.exists("data/processed/id2pos.json")
)

True
True


In [32]:
with open(
    "data/processed/pos2id.json",
    "r",
    encoding="utf-8"
) as f:
    loaded_pos2id = json.load(f)

with open(
    "data/processed/id2pos.json",
    "r",
    encoding="utf-8"
) as f:
    loaded_id2pos = json.load(f)

print("Loaded pos2id:")
print(loaded_pos2id)

print("\nLoaded id2pos:")
print(loaded_id2pos)

Loaded pos2id:
{'abb': 0, 'adj': 1, 'adv': 2, 'conj': 3, 'fw': 4, 'int': 5, 'n': 6, 'num': 7, 'part': 8, 'ppm': 9, 'pron': 10, 'punc': 11, 'sb': 12, 'tn': 13, 'v': 14}

Loaded id2pos:
{'0': 'abb', '1': 'adj', '2': 'adv', '3': 'conj', '4': 'fw', '5': 'int', '6': 'n', '7': 'num', '8': 'part', '9': 'ppm', '10': 'pron', '11': 'punc', '12': 'sb', '13': 'tn', '14': 'v'}


In [33]:
print(
    "Number of POS labels:",
    len(loaded_pos2id)
)

Number of POS labels: 15


In [34]:
from collections import Counter

pos_counter = Counter()

for record in records:
    pos_counter.update(
        record["pos_tags"]
    )

pos_distribution = pd.DataFrame(
    pos_counter.most_common(),
    columns=["pos_tag", "count"]
)

print(pos_distribution)

   pos_tag   count
0     part  133321
1        n  121959
2      ppm   85616
3        v   83054
4     punc   52781
5     pron   20008
6     conj   17760
7      adj   16083
8      adv   10534
9      num    5925
10      tn    5782
11      fw    3221
12     int     664
13     abb     360
14      sb     272


In [35]:
pos_distribution.to_csv(
    "reports/pos_distribution.csv",
    index=False,
    encoding="utf-8-sig"
)

print("POS distribution report saved!")

POS distribution report saved!


In [36]:
print(
    os.path.exists(
        "reports/pos_distribution.csv"
    )
)

True


12. Create Small Samples

In [37]:
rng = np.random.default_rng(42)

In [38]:
sample_100_indices = rng.choice(
    len(train_data),
    size=min(100, len(train_data)),
    replace=False
)

sample_100 = [
    train_data[i]
    for i in sample_100_indices
]

print(
    "sample_100 size:",
    len(sample_100)
)

sample_100 size: 100


In [39]:
sample_500_indices = rng.choice(
    len(train_data),
    size=min(500, len(train_data)),
    replace=False
)

sample_500 = [
    train_data[i]
    for i in sample_500_indices
]

print(
    "sample_500 size:",
    len(sample_500)
)

sample_500 size: 500


In [40]:
save_jsonl(
    sample_100,
    "data/samples/sample_100.jsonl"
)

save_jsonl(
    sample_500,
    "data/samples/sample_500.jsonl"
)

print("General sample datasets saved!")

General sample datasets saved!


In [41]:
segmentation_sample_100 = []

for record in sample_100:
    segmentation_sample_100.append({
        "id": record["id"],
        "text": record["text"],
        "tokens": record["tokens"],
        "seg_units": record["seg_units"],
        "seg_labels": record["seg_labels"]
    })

save_jsonl(
    segmentation_sample_100,
    "data/samples/segmentation_sample_100.jsonl"
)

print(
    "Segmentation sample saved:",
    len(segmentation_sample_100)
)

Segmentation sample saved: 100


In [42]:
pos_sample_100 = []

for record in sample_100:
    pos_sample_100.append({
        "id": record["id"],
        "text": record["text"],
        "tokens": record["tokens"],
        "pos_tags": record["pos_tags"],
        "pos_ids": record["pos_ids"]
    })

save_jsonl(
    pos_sample_100,
    "data/samples/pos_sample_100.jsonl"
)

print(
    "POS sample saved:",
    len(pos_sample_100)
)

POS sample saved: 100


In [43]:
def load_jsonl(path):
    data = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:
            data.append(
                json.loads(line)
            )

    return data

In [44]:
test_train_load = load_jsonl(
    "data/processed/train.jsonl"
)

print(
    "Loaded train records:",
    len(test_train_load)
)

Loaded train records: 33537


In [45]:
test_val_load = load_jsonl(
    "data/processed/validation.jsonl"
)

print(
    "Loaded validation records:",
    len(test_val_load)
)

Loaded validation records: 4192


In [46]:
test_test_load = load_jsonl(
    "data/processed/test.jsonl"
)

print(
    "Loaded test records:",
    len(test_test_load)
)

Loaded test records: 4193


In [47]:
def validate_dataset(data, name):
    errors = 0

    for record in data:

        if (
            len(record["tokens"])
            != len(record["pos_tags"])
        ):
            errors += 1

        if (
            len(record["tokens"])
            != len(record["pos_ids"])
        ):
            errors += 1

        if (
            len(record["seg_units"])
            != len(record["seg_labels"])
        ):
            errors += 1

        if (
            "".join(record["seg_units"])
            != record["text"]
        ):
            errors += 1

    print(
        name,
        "validation errors:",
        errors
    )

In [48]:
validate_dataset(
    train_data,
    "Train"
)

validate_dataset(
    validation_data,
    "Validation"
)

validate_dataset(
    test_data,
    "Test"
)

Train validation errors: 0
Validation validation errors: 0
Test validation errors: 0


In [49]:
train_texts = {
    record["text"]
    for record in train_data
}

validation_texts = {
    record["text"]
    for record in validation_data
}

test_texts = {
    record["text"]
    for record in test_data
}

print(
    "Train ↔ Validation:",
    len(train_texts & validation_texts)
)

print(
    "Train ↔ Test:",
    len(train_texts & test_texts)
)

print(
    "Validation ↔ Test:",
    len(validation_texts & test_texts)
)

Train ↔ Validation: 0
Train ↔ Test: 0
Validation ↔ Test: 0


In [50]:
missing_count = 0

for record in records:

    required_fields = [
        "id",
        "text",
        "tokens",
        "pos_tags",
        "pos_ids",
        "seg_units",
        "seg_labels"
    ]

    for field in required_fields:
        if field not in record:
            missing_count += 1

print(
    "Missing required fields:",
    missing_count
)

Missing required fields: 0


In [51]:
empty_texts = 0
empty_tokens = 0
empty_pos = 0

for record in records:

    if not record["text"]:
        empty_texts += 1

    if len(record["tokens"]) == 0:
        empty_tokens += 1

    if len(record["pos_tags"]) == 0:
        empty_pos += 1

print("Empty texts:", empty_texts)
print("Empty token lists:", empty_tokens)
print("Empty POS lists:", empty_pos)

Empty texts: 0
Empty token lists: 0
Empty POS lists: 0


In [52]:
seg_counter = Counter()

for record in records:
    seg_counter.update(
        record["seg_labels"]
    )

print(
    "Segmentation label distribution:"
)

for label, count in seg_counter.items():
    print(label, count)

Segmentation label distribution:
B 557340
I 828191


In [53]:
seg_distribution = pd.DataFrame(
    seg_counter.items(),
    columns=[
        "segmentation_label",
        "count"
    ]
)

seg_distribution

,segmentation_label,count
0,B,557340
1,I,828191


In [54]:
seg_distribution.to_csv(
    "reports/segmentation_distribution.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "Segmentation distribution saved!"
)

Segmentation distribution saved!


In [55]:
for record in train_data[:3]:

    print("=" * 60)

    print(
        "ID:",
        record["id"]
    )

    print(
        "Text:",
        record["text"]
    )

    print(
        "Tokens:",
        record["tokens"]
    )

    print(
        "POS:",
        record["pos_tags"]
    )

    print(
        "POS IDs:",
        record["pos_ids"]
    )

    print(
        "Seg labels:",
        record["seg_labels"][:30]
    )

ID: sent_05161
Text: နေရာများစွာမှရရှိထားချက်အရ၊70%သောကမ္ဘာ့မွတ်စလင်များသည်ဆွန်နီ၊20%သည်၊ရှီးအတ်နှင့်ကျန်10%သည်အခြားသောအမျိုးမျိုးရှိကြသည်အဖွဲ့ငယ်များနှင့်အစ္စလာမ်အခွဲများဖြစ်သည်။
Tokens: ['နေရာ', 'များ', 'စွာ', 'မှ', 'ရရှိ', 'ထား', 'ချက်', 'အရ', '၊', '70', '%', 'သော', 'ကမ္ဘာ့', 'မွတ်စလင်', 'များ', 'သည်', 'ဆွန်နီ', '၊', '20', '%', 'သည်', '၊', 'ရှီးအတ်', 'နှင့်', 'ကျန်', '10', '%', 'သည်', 'အခြား', 'သော', 'အမျိုးမျိုး', 'ရှိ', 'ကြ', 'သည်', 'အဖွဲ့', 'ငယ်', 'များ', 'နှင့်', 'အစ္စလာမ်', 'အခွဲ', 'များ', 'ဖြစ်', 'သည်', '။']
POS: ['n', 'adj', 'part', 'ppm', 'v', 'part', 'part', 'ppm', 'punc', 'fw', 'sb', 'part', 'n', 'n', 'part', 'ppm', 'n', 'punc', 'fw', 'sb', 'ppm', 'punc', 'n', 'conj', 'v', 'fw', 'sb', 'ppm', 'adj', 'part', 'n', 'v', 'part', 'ppm', 'n', 'adj', 'part', 'conj', 'n', 'n', 'part', 'v', 'ppm', 'punc']
POS IDs: [6, 1, 8, 9, 14, 8, 8, 9, 11, 4, 12, 8, 6, 6, 8, 9, 6, 11, 4, 12, 9, 11, 6, 3, 14, 4, 12, 9, 1, 8, 6, 14, 8, 9, 6, 1, 8, 3, 6, 6, 8, 14, 9, 11]
Seg labels: ['B', 'I', 'I',

13. Preprocessing Summary

In [56]:
print("===== SHWE MYANMAR PREPROCESSING SUMMARY =====")

print("Final clean records:", len(records))

print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

print("POS labels:", len(pos2id))

print(
    "Segmentation labels:",
    sorted(seg_counter.keys())
)

print(
    "Train/Validation overlap:",
    len(train_texts & validation_texts)
)

print(
    "Train/Test overlap:",
    len(train_texts & test_texts)
)

print(
    "Validation/Test overlap:",
    len(validation_texts & test_texts)
)

===== SHWE MYANMAR PREPROCESSING SUMMARY =====
Final clean records: 41922
Train: 33537
Validation: 4192
Test: 4193
POS labels: 15
Segmentation labels: ['B', 'I']
Train/Validation overlap: 0
Train/Test overlap: 0
Validation/Test overlap: 0


In [57]:
report = f"""# Shwe Myanmar Dataset Preprocessing Report

## Dataset

Dataset: myPOS Version 3.0

## Preprocessing

- Loaded Burmese text using UTF-8.
- Parsed word/POS annotations.
- Preserved compound-word information.
- Normalized text using Unicode NFC.
- Validated token/POS alignment.
- Standardized the POS label set.
- Created POS-to-ID and ID-to-POS mappings.
- Created B/I word-segmentation labels.
- Used Unicode grapheme clusters with regex \\\\X.
- Removed duplicate records.
- Removed duplicated sentence texts before splitting.
- Split the final dataset into train/validation/test.
- Used random seed 42.

## Final Dataset

Final clean records: {len(records)}

Train records: {len(train_data)}

Validation records: {len(validation_data)}

Test records: {len(test_data)}

Number of POS classes: {len(pos2id)}

## Segmentation

B = Beginning of a word

I = Inside a word

## Validation

Train/Validation overlap: {len(train_texts & validation_texts)}

Train/Test overlap: {len(train_texts & test_texts)}

Validation/Test overlap: {len(validation_texts & test_texts)}

## Output Files

- train.jsonl
- validation.jsonl
- test.jsonl
- pos2id.json
- id2pos.json
- sample_100.jsonl
- sample_500.jsonl
- segmentation_sample_100.jsonl
- pos_sample_100.jsonl
- pos_distribution.csv
- segmentation_distribution.csv
"""

with open(
    "reports/preprocessing_report.md",
    "w",
    encoding="utf-8"
) as f:
    f.write(report)

print(
    "Preprocessing report created!"
)

Preprocessing report created!


In [58]:
for root, dirs, files in os.walk("."):

    if (
        root.startswith("./data")
        or root.startswith("./reports")
    ):

        for file in files:
            path = os.path.join(
                root,
                file
            )

            print(path)

./data/samples/pos_sample_100.jsonl
./data/samples/sample_100.jsonl
./data/samples/segmentation_sample_100.jsonl
./data/samples/sample_500.jsonl
./data/processed/validation.jsonl
./data/processed/id2pos.json
./data/processed/pos2id.json
./data/processed/test.jsonl
./data/processed/train.jsonl
./reports/pos_distribution.csv
./reports/segmentation_distribution.csv
./reports/preprocessing_report.md


In [59]:
!zip -r shwemyanmar_member1_output.zip data reports

  adding: data/ (stored 0%)
  adding: data/samples/ (stored 0%)
  adding: data/samples/pos_sample_100.jsonl (deflated 83%)
  adding: data/samples/sample_100.jsonl (deflated 85%)
  adding: data/samples/segmentation_sample_100.jsonl (deflated 85%)
  adding: data/samples/sample_500.jsonl (deflated 86%)
  adding: data/processed/ (stored 0%)
  adding: data/processed/validation.jsonl (deflated 86%)
  adding: data/processed/id2pos.json (deflated 50%)
  adding: data/processed/pos2id.json (deflated 43%)
  adding: data/processed/test.jsonl (deflated 86%)
  adding: data/processed/train.jsonl (deflated 86%)
  adding: reports/ (stored 0%)
  adding: reports/pos_distribution.csv (deflated 18%)
  adding: reports/segmentation_distribution.csv (stored 0%)
  adding: reports/preprocessing_report.md (deflated 52%)


In [62]:
from google.colab import files

files.download(
    "shwemyanmar_member1_output.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>